# Lección 03 - Patrones de Diseño Agénticos

En esta lección, exploramos tres patrones de diseño fundamentales para construir agentes de IA efectivos:

1. **Instrucciones Claras para el Agente** — Crear indicaciones precisas que definan el rol y guíen el comportamiento del agente  
2. **Salida Estructurada con Modelos Pydantic** — Asegurar que los agentes devuelvan datos predecibles y validados  
3. **Agentes de Responsabilidad Única** — Diseñar agentes enfocados que hagan bien una sola tarea

Aplicaremos cada patrón a un escenario de **recomendador de destinos de viaje**, construyendo progresivamente un sistema que pueda sugerir destinos, verificar disponibilidad y manejar la logística.


## Configuración


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Patrón 1: Instrucciones Claras para el Agente

El patrón más efectivo es también el más simple: escribir instrucciones claras y detalladas para tu agente.

Buenas instrucciones definen:
- **Quién** es el agente (persona y tono)
- **Qué** debe hacer (responsabilidades paso a paso)
- **Cómo** debe comportarse (restricciones y estilo)

A continuación, creamos un agente concierge de viajes con instrucciones explícitas que moldean cada respuesta que produce.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Patrón 2: Salida Estructurada con Modelos Pydantic

El texto de forma libre es útil para la conversación, pero los sistemas posteriores necesitan datos estructurados.  
Al combinar **modelos Pydantic** con una **función de herramienta**, podemos:

- Definir un esquema exacto para la salida del agente  
- Validar respuestas automáticamente  
- Integrar los resultados del agente en la lógica de la aplicación de forma confiable  

También presentamos una herramienta que devuelve detalles del destino para que el agente fundamente sus recomendaciones en datos reales.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Patrón 3: Agentes de Responsabilidad Única

Las tareas complejas se benefician de dividir el trabajo entre múltiples agentes enfocados, cada uno con una única responsabilidad:

- Un **Experto en Destinos** que conoce sobre lugares y disponibilidad
- Un **Planificador de Logística** que se encarga de vuelos, hoteles y itinerarios

Esto refleja el principio de ingeniería de software de *separación de responsabilidades* — cada agente es más fácil de probar, mantener y mejorar de forma independiente.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Resumen

En esta lección aplicamos tres patrones de diseño agente a un escenario de recomendación de viajes:

| Patrón | Idea clave | Beneficio |
|---|---|---|
| **Instrucciones Claras** | Definir persona, responsabilidades y restricciones desde el principio | Comportamiento del agente consistente y acorde a la marca |
| **Salida Estructurada** | Usar modelos Pydantic como formato de respuesta | Resultados validados y legibles por máquina |
| **Responsabilidad Única** | Dar a cada agente un trabajo enfocado | Más fácil de probar, mantener y componer |

Estos patrones se componen de forma natural: puedes combinar instrucciones claras con salida estructurada dentro de un agente de responsabilidad única para construir sistemas robustos y listos para producción.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por lograr precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por humanos. No nos hacemos responsables por malentendidos o interpretaciones erróneas que puedan surgir del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
